In [3]:
#3.1
import pandas as pd
from sklearn.model_selection import train_test_split

# Datensatz laden (mit Semikolon als Trenner)
df = pd.read_csv("Daten LB.csv", sep=';')

# 3.1 Teilen Sie Ihren Datensatz in einen test- und einen train-Satz ein
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
#Test
print(f"Datensatz erfolgreich geteilt: {len(train_df)} Trainingsdaten, {len(test_df)} Testdaten.")

Datensatz erfolgreich geteilt: 6992 Trainingsdaten, 1748 Testdaten.


In [5]:
#3.2
y_ftr_train = train_df["FTR"]
y_ftr_test = test_df["FTR"]

y_goals_train = train_df[["FTHG", "FTAG"]]
y_goals_test = test_df[["FTHG", "FTAG"]]

drop_cols = ["FTR", "FTHG", "FTAG"]

X_train = train_df.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)

X_train = X_train.select_dtypes(include=["number"])
X_test = X_test.select_dtypes(include=["number"])

#Modelle erstellen

# Klassifikation (Spielausgang)
model_ftr = RandomForestClassifier(n_estimators=10, random_state=42)
model_ftr.fit(X_train, y_ftr_train)

# Regression (Tore)
model_goals = RandomForestRegressor(n_estimators=10, random_state=42)
model_goals.fit(X_train, y_goals_train)

print("Modelle wurden erfolgreich auf den Trainingsdaten berechnet.")

Modelle wurden erfolgreich auf den Trainingsdaten berechnet.


Ich habe Random Forest gewählt, weil er gut mit den komplexen und nicht linearen Zusammenhängen in meinen Fussballdaten umgehen kann. Er ist auch robust gegenüber zufälligen Schwankungen. Er kann auch den Spielausgang wie auch die Toranzahl vorhersagen. Das Random Forest-Modell habe ich auch am besten verstanden und machte für mich am meisten Sinn weil ich damit Wahrscheinlichkeiten ausrechnen kann. Für mich war es ideal als einfache und stabile Grundlage für das erste Modelle.

In [7]:
#3.3
import numpy as np

random_indices = np.random.choice(X_test.index, size=10, replace=False)

for idx in random_indices:
    game_features = X_test.loc[[idx]]
    
    real_info = df.loc[idx]
    
    pred_ftr = model_ftr.predict(game_features)[0]
    prob_ftr = model_ftr.predict_proba(game_features)[0]
    pred_goals = model_goals.predict(game_features)[0]

#Überprüfung der Vorhersagen
    print(f"Spiel: {real_info['HomeTeam']} vs {real_info['AwayTeam']}")
    print(f"  Tatsächlich: {real_info['FTR']} (Tore: {real_info['FTHG']} : {real_info['FTAG']})")
    print(f"  Vorhersage:  {pred_ftr} (Erwartete Tore: {pred_goals[0]:.1f} : {pred_goals[1]:.1f})")
    print(f"  Siegchancen: Heim {prob_ftr[2]:.1%}, Remis {prob_ftr[1]:.1%}, Gast {prob_ftr[0]:.1%}")
    print("-" * 50)

    correct = 0

for idx in random_indices:
    game_features = X_test.loc[[idx]]
    real_info = df.loc[idx]

    pred_ftr = model_ftr.predict(game_features)[0]

    if pred_ftr == real_info["FTR"]:
        correct += 1

accuracy = correct / len(random_indices)

print(f"Vorhersage Tore richtig: {accuracy:.2%}")

correct_prob = 0

for idx in random_indices:
    game_features = X_test.loc[[idx]]
    real_info = df.loc[idx]

    prob = model_ftr.predict_proba(game_features)[0]
    pred_class = model_ftr.predict(game_features)[0]

    # richtige Klasse finden
    classes = model_ftr.classes_
    true_index = list(classes).index(real_info["FTR"])

    # war die höchste Wahrscheinlichkeit korrekt?
    if prob.argmax() == true_index:
        correct_prob += 1

prob_accuracy = correct_prob / len(random_indices)

print(f"H/R/G Vorhersage Richtig: {prob_accuracy:.2%}")

Spiel: Liverpool vs Sunderland
  Tatsächlich: H (Tore: 2 : 0)
  Vorhersage:  H (Erwartete Tore: 1.6 : 0.2)
  Siegchancen: Heim 60.0%, Remis 40.0%, Gast 0.0%
--------------------------------------------------
Spiel: West Ham vs Burnley
  Tatsächlich: D (Tore: 1 : 1)
  Vorhersage:  D (Erwartete Tore: 0.9 : 1.1)
  Siegchancen: Heim 30.0%, Remis 50.0%, Gast 20.0%
--------------------------------------------------
Spiel: Man City vs West Ham
  Tatsächlich: H (Tore: 2 : 0)
  Vorhersage:  A (Erwartete Tore: 1.9 : 0.3)
  Siegchancen: Heim 40.0%, Remis 20.0%, Gast 40.0%
--------------------------------------------------
Spiel: Aston Villa vs Fulham
  Tatsächlich: H (Tore: 1 : 0)
  Vorhersage:  D (Erwartete Tore: 1.4 : 0.5)
  Siegchancen: Heim 40.0%, Remis 50.0%, Gast 10.0%
--------------------------------------------------
Spiel: Middlesbrough vs West Ham
  Tatsächlich: H (Tore: 2 : 0)
  Vorhersage:  H (Erwartete Tore: 2.6 : 0.1)
  Siegchancen: Heim 100.0%, Remis 0.0%, Gast 0.0%
---------------

Wenn ich die Vorhersagen drucken lasse und dann mit den richtigen Resultaten vergleiche komme ich zum Entschluss, dass 60 % der Vorhersagen stimmen. Dies wird wahrscheinlich daran liegen, dass mein Datensatz wichtige Sachen, die für genauere Vorhersagen wichtig wären nicht beinhaltet. Z.B. sehe ich nicht welche Spieler verletzt sind, was einen grossen Unterschied auf die Ergebnisse machen kann. Wenn z.B. der erste Torhüter einer Mannschaft auf dem Feld steht, geht nur 1 von 14 Schüssen ins Tor, wenn aber allerdings der Ersatztorhüter spielt gehen vielleicht 3 Schüsse ins Tor, was ein anderes Ergebnis ist als mein Datensatz üblicherweise mit den Daten berechnen würde.

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model_ftr.predict(X_test)
acc = accuracy_score(y_ftr_test, y_pred)

print("Accuracy:", acc)